# Building a Payment-Aware AI Agent with x402 and Claude

This cookbook shows how to give Claude instant awareness of the x402 payment ecosystem using a remote MCP server — **no API keys, no setup, just a URL**.

In roughly 30 seconds of configuration, your Claude agent gains the ability to:
- Discover 250+ live APIs that accept micropayments
- Look up service details, pricing, and uptime
- Verify trust via cryptographic EdDSA attestations
- Route payment requests intelligently across the ecosystem

---

## What is x402?

[x402](https://github.com/coinbase/x402) is an open HTTP payment standard originally proposed by Coinbase. When an AI agent calls an API that requires payment, the server returns HTTP `402 Payment Required` with a machine-readable payment header. The agent pays (typically fractions of a cent in USDC on Base), and the server proceeds.

This is significant for agentic AI: it enables **fully autonomous API monetization** without OAuth flows, billing accounts, or human intervention. An agent can discover a service, verify it, and pay for exactly what it uses — all in a single HTTP exchange.

## What is the x402 Discovery API?

The [x402 Service Discovery API](https://x402-discovery-api.onrender.com) is a free, open-source registry that continuously scans and indexes x402-enabled services. It exposes six tools via MCP:

| Tool | Description |
|------|-------------|
| `discover_services` | List and filter x402-enabled APIs by category, price, uptime |
| `search_services` | Semantic search across service names and descriptions |
| `get_service_details` | Full details: endpoints, pricing, facilitator compatibility |
| `list_categories` | Browse service categories in the ecosystem |
| `get_attestation` | Fetch a signed JWT attestation proving a service's identity |
| `verify_attestation` | Verify an attestation's cryptographic signature |

---

## Prerequisites

- An Anthropic API key ([get one here](https://console.anthropic.com/settings/keys))
- Python 3.9+
- No wallet, no blockchain setup, no x402 service account — the discovery API is free and unauthenticated

## Installation

In [ ]:
%pip install anthropic httpx

## Setup

Set your Anthropic API key. The x402 Discovery API requires no authentication.

In [ ]:
import os
import json
import anthropic
import httpx

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
assert ANTHROPIC_API_KEY, "Set ANTHROPIC_API_KEY environment variable"

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

X402_MCP_URL = "https://x402-discovery-api.onrender.com/mcp/"
print(f"Using x402 Discovery MCP server: {X402_MCP_URL}")

## Two Integration Paths

### Path A — Claude Desktop (30-second setup)

Add this to your `claude_desktop_config.json`:

```json
{
  "mcpServers": {
    "x402-discovery": {
      "url": "https://x402-discovery-api.onrender.com/mcp/",
      "type": "http"
    }
  }
}
```

After restarting Claude Desktop, you can ask: *"What x402 APIs are available for weather data?"* and Claude will use the MCP tools automatically.

### Path B — Programmatic (this notebook)

We'll call the x402 Discovery API directly via its REST endpoints and pass results to Claude. This approach works in any Python environment and gives full control over tool orchestration.

---
## Demo 1: Discover Available x402 Services

We query the discovery API and let Claude summarize what's available.

In [ ]:
def discover_services(limit: int = 20, min_uptime: float = 0.8) -> dict:
    """Call the x402 Discovery API to list services."""
    resp = httpx.get(
        "https://x402-discovery-api.onrender.com/v1/services",
        params={"limit": limit, "min_uptime": min_uptime},
        timeout=15,
    )
    resp.raise_for_status()
    return resp.json()


services_data = discover_services(limit=20)
print(f"Total services indexed: {services_data.get('total', 'N/A')}")

In [ ]:
# Let Claude interpret the ecosystem for us
services_json = json.dumps(services_data, indent=2)[:8000]  # trim for context

response = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": (
                "Here is data from the x402 Service Discovery API — a registry of APIs\n"
                "that accept HTTP 402 micropayments. Summarize what kinds of services\n"
                "are available and which look most useful for AI agents.\n\n"
                f"{services_json}"
            ),
        }
    ],
)
print(response.content[0].text)

---
## Demo 2: Payment-Aware Tool Use

Now we give Claude the discovery tools directly so it can autonomously look up services, check details, and verify trust — simulating what a fully agentic payment-aware Claude would do.

In [ ]:
X402_TOOLS = [
    {
        "name": "discover_services",
        "description": (
            "Discover x402-enabled APIs in the registry. Returns a list of services\n"
            "with their URLs, pricing, categories, uptime, and trust scores.\n"
            "Use this when you need to find APIs that accept micropayments."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "limit": {"type": "integer", "description": "Max results (default 20)"},
                "category": {"type": "string", "description": "Filter by category (e.g. 'data', 'ai')"},
                "min_uptime": {"type": "number", "description": "Minimum uptime ratio (0.0-1.0)"},
                "min_trust_score": {"type": "number", "description": "Minimum trust score (0.0-1.0)"},
            },
        },
    },
    {
        "name": "search_services",
        "description": "Search x402 services by keyword across names and descriptions.",
        "input_schema": {
            "type": "object",
            "properties": {
                "q": {"type": "string", "description": "Search query"},
                "limit": {"type": "integer", "description": "Max results"},
            },
            "required": ["q"],
        },
    },
    {
        "name": "get_service_details",
        "description": "Get full details for a specific x402 service: endpoints, pricing, facilitators.",
        "input_schema": {
            "type": "object",
            "properties": {
                "service_id": {"type": "string", "description": "Service ID from discover/search results"},
            },
            "required": ["service_id"],
        },
    },
    {
        "name": "get_attestation",
        "description": (
            "Fetch a signed EdDSA attestation JWT for a service.\n"
            "An attestation cryptographically proves the service's identity and\n"
            "is useful for agents that need trustless verification before paying."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "service_id": {"type": "string", "description": "Service ID to attest"},
            },
            "required": ["service_id"],
        },
    },
]


def call_x402_tool(tool_name: str, tool_input: dict) -> str:
    """Execute an x402 Discovery API tool call."""
    base_url = "https://x402-discovery-api.onrender.com"
    try:
        if tool_name == "discover_services":
            resp = httpx.get(f"{base_url}/v1/services", params=tool_input, timeout=15)
        elif tool_name == "search_services":
            resp = httpx.get(f"{base_url}/v1/search", params=tool_input, timeout=15)
        elif tool_name == "get_service_details":
            sid = tool_input["service_id"]
            resp = httpx.get(f"{base_url}/v1/services/{sid}", timeout=15)
        elif tool_name == "get_attestation":
            sid = tool_input["service_id"]
            resp = httpx.get(f"{base_url}/v1/attest/{sid}", timeout=15)
        else:
            return json.dumps({"error": f"Unknown tool: {tool_name}"})
        resp.raise_for_status()
        return json.dumps(resp.json())
    except Exception as e:
        return json.dumps({"error": str(e)})

print("Tools defined:", [t['name'] for t in X402_TOOLS])

In [ ]:
def run_x402_agent(user_query: str, max_turns: int = 5) -> str:
    """
    Run Claude as a payment-aware agent with access to x402 discovery tools.
    Claude will autonomously decide which tools to call and in what order.
    """
    messages = [{"role": "user", "content": user_query}]
    system = (
        "You are a payment-aware AI agent with access to the x402 Service Discovery API. "
        "x402 is an HTTP micropayment standard: services return HTTP 402 and agents pay "
        "in USDC fractions of a cent to unlock access. Use the discovery tools to help "
        "users find, evaluate, and trust x402-enabled APIs for their use case. "
        "Always check service quality (trust score, uptime) before recommending one."
    )

    for turn in range(max_turns):
        response = client.messages.create(
            model="claude-opus-4-5",
            max_tokens=2048,
            system=system,
            tools=X402_TOOLS,
            messages=messages,
        )

        # Collect any text output
        for block in response.content:
            if hasattr(block, 'text'):
                print(f"Claude: {block.text}")

        # If Claude is done, return
        if response.stop_reason == "end_turn":
            break

        # Process tool calls
        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  → Calling tool: {block.name}({block.input})")
                    result = call_x402_tool(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })

            # Add assistant response + tool results to history
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})
        else:
            break

    return "Agent loop complete"


# Run the agent
run_x402_agent(
    "I need a weather or geolocation API for my agent. Find the best x402 option — "
    "check its trust score and get an attestation so I can verify it's legitimate."
)

---
## Demo 3: Verify Trust Before Paying

Before an agent pays for a service, it can verify the service's identity cryptographically using EdDSA attestations. This is especially important in autonomous settings where no human reviews the transaction.

The attestation flow:
1. Agent discovers a service
2. Agent fetches an attestation JWT from the discovery API
3. Agent verifies the JWT using the discovery API's public key (JWKS endpoint)
4. If valid → proceed with payment; if invalid → reject

In [ ]:
import base64


def verify_service_attestation(service_id: str) -> dict:
    """
    Fetch and verify a cryptographic attestation for an x402 service.
    Returns verification status and decoded claims.
    """
    base_url = "https://x402-discovery-api.onrender.com"

    # 1. Fetch the attestation JWT
    attest_resp = httpx.get(f"{base_url}/v1/attest/{service_id}", timeout=15)
    if attest_resp.status_code != 200:
        return {"verified": False, "error": f"No attestation found (HTTP {attest_resp.status_code})"}
    jwt_token = attest_resp.json().get("attestation")
    if not jwt_token:
        return {"verified": False, "error": "No attestation in response"}

    # 2. Decode header and payload (no signature verification here — use PyJWT for production)
    try:
        parts = jwt_token.split(".")
        def decode_b64(s):
            s += "=" * (-len(s) % 4)
            return json.loads(base64.urlsafe_b64decode(s))
        header = decode_b64(parts[0])
        payload = decode_b64(parts[1])
        return {
            "verified": True,
            "algorithm": header.get("alg"),
            "service_url": payload.get("serviceUrl"),
            "uptime": payload.get("uptimeRatio"),
            "trust_score": payload.get("trustScore"),
            "issued_at": payload.get("iat"),
            "expires_at": payload.get("exp"),
            "issuer": payload.get("iss"),
        }
    except Exception as e:
        return {"verified": False, "error": str(e)}


# Get the first available service and verify it
services = discover_services(limit=5)
first_service = services["services"][0] if services.get("services") else None

if first_service:
    sid = first_service["id"]
    print(f"Verifying attestation for service: {first_service['url']}\n")
    result = verify_service_attestation(sid)
    print(json.dumps(result, indent=2))
else:
    print("No services returned by the API")

> **Note:** In production, use `PyJWT` with the discovery API's JWKS endpoint
> (`https://x402-discovery-api.onrender.com/jwks`) for full signature verification:
> ```python
> import jwt
> from jwt import PyJWKClient
> jwks_client = PyJWKClient("https://x402-discovery-api.onrender.com/jwks")
> signing_key = jwks_client.get_signing_key_from_jwt(token)
> claims = jwt.decode(token, signing_key.key, algorithms=["EdDSA"])
> ```

---
## Claude Desktop Configuration

To use the x402 discovery tools interactively in Claude Desktop, add the following to your configuration file.

**macOS:** `~/Library/Application Support/Claude/claude_desktop_config.json`
**Windows:** `%APPDATA%\\Claude\\claude_desktop_config.json`

```json
{
  "mcpServers": {
    "x402-discovery": {
      "url": "https://x402-discovery-api.onrender.com/mcp/",
      "type": "http"
    }
  }
}
```

Then try prompts like:
- *"Find me the best x402 API for financial data"*
- *"What x402 services have >99% uptime and cost less than $0.01 per call?"*
- *"Get an attestation for service ID xyz and tell me if it's trustworthy"*

---
## Resources

| Resource | Link |
|----------|------|
| x402 Discovery API | https://x402-discovery-api.onrender.com |
| MCP Server URL | https://x402-discovery-api.onrender.com/mcp/ |
| JWKS (for attestation verification) | https://x402-discovery-api.onrender.com/jwks |
| Source code | https://github.com/rplryan/x402-discovery-mcp |
| x402 Standard (Coinbase) | https://github.com/coinbase/x402 |
| PyPI package | https://pypi.org/project/x402-payment-harness/ |

## About x402 Discovery

The x402 Service Discovery API is an open-source project that continuously monitors and indexes x402-enabled services. It is free to use, requires no authentication, and provides:

- **251+ indexed services** with auto-scanning every 6 hours
- **Trust scoring** based on uptime, response time, and facilitator compatibility
- **EdDSA attestations** — RFC 7517 JWKS-backed cryptographic proofs of service identity
- **MCP-native tools** for direct integration with Claude and other MCP-compatible agents